In [ ]:
!pip install openml scikit-learn xgboost scipy numpy matplotlib seaborn -q
!pip install cudf-cu12 cuml-cu12 --extra-index-url=https://pypi.nvidia.com -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 44.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 MB 214.3 MB/s eta 0:00:00


In [ ]:
# ============================================================
# NEW INVENTIONS: RWC & GWL (MANIFOLD FORESTS)
# Fully Optimized for T4 GPU Accuracy & Speed
# Authors: Debanik Debnath + Xylia
# ============================================================

import subprocess, sys, time, warnings
import numpy as np
import openml
import cupy as cp
import cuml
from cupyx import scatter_add
from cuml.neighbors import NearestNeighbors as cuNN
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from scipy.sparse import csr_matrix, diags

warnings.filterwarnings("ignore")
print(f"✓ GPU detected: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")
print(f"  VRAM: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB")

# ── DATA LOADING ──
print("\nLoading EEG Eye State corpus (OpenML 1471)...")
dataset = openml.datasets.get_dataset(1471)
X_raw, y_raw, _, _ = dataset.get_data(target=dataset.default_target_attribute)
X_raw = X_raw.values.astype(np.float64)
le = LabelEncoder()
y_raw = le.fit_transform(y_raw.astype(int).values)

# EEG Preprocessing (Bipolar Montage + Spectral + Robust Scaling)
def bipolar_montage(X):
    X = np.clip(X, -15, 15)
    X_diff = X[:, :-1] - X[:, 1:]
    X_coh  = np.var(X, axis=1, keepdims=True)
    return np.hstack([X, X_diff, X_coh])

def spectral_transform(X, n_bins=50):
    return np.abs(np.fft.rfft(X, axis=1))[:, :n_bins]

scaler = RobustScaler(quantile_range=(15.0, 85.0))
X_bip = bipolar_montage(X_raw)
X_spec = spectral_transform(X_raw)
X_processed = scaler.fit_transform(np.hstack([X_bip, X_spec]))
print(f"  Processed Shape (with Spectral): {X_processed.shape}")

# ── RWC IMPLEMENTATION ──
class RiemannianWaveClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_components=128, k_neighbors=15, n_freq=30,
                 epsilon=0.1, potential_strength=15.0, diffusion_time=0.5,
                 hrf_freq=30.0, hrf_gamma=10.0):
        self.n_components = n_components
        self.k_neighbors = k_neighbors
        self.n_freq = n_freq
        self.epsilon = epsilon
        self.potential_strength = potential_strength
        self.diffusion_time = diffusion_time
        self.hrf_freq = hrf_freq
        self.hrf_gamma = hrf_gamma

    def _build_manifold(self, X):
        N = len(X)
        k = min(self.k_neighbors, N - 1)
        X_gpu = cp.asarray(X, dtype=cp.float32)
        nbrs = cuNN(n_neighbors=k, metric='euclidean')
        nbrs.fit(X_gpu)
        dists_gpu, indices_gpu = nbrs.kneighbors(X_gpu)

        sigma_i = dists_gpu[:, -1]
        sigma_j = sigma_i[indices_gpu]

        W_dense = cp.exp(-dists_gpu**2 / (sigma_i[:, None] * sigma_j + 1e-12))
        W = cp.zeros((N, N), dtype=cp.float32)
        scatter_add(W, (cp.arange(N)[:, None], indices_gpu), W_dense)
        W = (W + W.T) / 2.0

        d = cp.sum(W, axis=1)
        d_i = cp.where(d > 0, 1.0 / cp.sqrt(d), 0.0)
        L = cp.eye(N) - (d_i[:, None] * W * d_i[None, :])

        vals, vecs = cp.linalg.eigh(L)
        return vecs[:, 1:self.n_components+1], vals[1:self.n_components+1]

    def _wave_energy_batch(self, phi_q, phi_c_train, mu_c):
        phi_q_g = cp.asarray(phi_q, dtype=cp.float32)
        phi_c_g = cp.asarray(phi_c_train, dtype=cp.float32)
        mu_c_g = cp.asarray(mu_c, dtype=cp.float32)

        freqs = cp.linspace(0.01, cp.max(cp.abs(mu_c_g)) + 1.0, self.n_freq)
        w_sq = freqs**2

        lor = self.epsilon / (cp.pi * ((w_sq[:, None] - cp.abs(mu_c_g[None, :]))**2 + self.epsilon**2))
        energies = cp.zeros((phi_q_g.shape[0],), dtype=cp.float32)
        batch_size = 500

        for i in range(0, phi_c_g.shape[0], batch_size):
            phi_c_batch = phi_c_g[i:i+batch_size]
            K_batch = cp.einsum('fm,qm,cm->qcf', lor, phi_q_g, phi_c_batch)
            energies += cp.sum(K_batch, axis=(1, 2))

        return cp.asnumpy(energies)

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.X_train_ = X
        self.y_train_ = y
        phi_g, lam_g = self._build_manifold(X)
        self.phi_ = cp.asnumpy(phi_g)

        self.potentials_ = []
        self.phi_class_ = {}
        for c in self.classes_:
            V_diag = cp.where(cp.asarray(y) == c, -self.potential_strength, self.potential_strength * 0.5)
            V_proj = cp.sum(V_diag[:, None] * phi_g**2, axis=0)
            self.potentials_.append(cp.asnumpy(lam_g + V_proj))
            self.phi_class_[c] = self.phi_[y == c]
        return self

    def predict(self, X):
        X_train_g, X_new_g = cp.asarray(self.X_train_, dtype=cp.float32), cp.asarray(X, dtype=cp.float32)

        # V13 Insight: Tighten the local search to k=5 to preserve sharp holography
        nbrs = cuNN(n_neighbors=5).fit(X_train_g)
        dists_g, idx_g = nbrs.kneighbors(X_new_g)
        dists, idx = cp.asnumpy(dists_g), cp.asnumpy(idx_g)
        B, m = len(X), self.phi_.shape[1]
        phi_q = np.zeros((B, m))

        for i in range(B):
            w_proj = np.exp(-dists[i]**2 / (2 * (dists[i].mean() + 1e-8)**2))
            phi_q[i] = (w_proj / (w_proj.sum() + 1e-12)) @ self.phi_[idx[i]]

        energies_gwl = np.zeros((B, len(self.classes_)))
        for ci, c in enumerate(self.classes_):
            energies_gwl[:, ci] = self._wave_energy_batch(phi_q, self.phi_class_[c], self.potentials_[ci])

        hrf_freq = self.hrf_freq
        hrf_gamma = self.hrf_gamma
        energies_hrf = np.zeros((B, len(self.classes_)))
        local_y = np.asarray(self.y_train_)[idx]

        for i in range(B):
            # EXACT V13 WAVE EQUATION: Restored d**2 (Removed d**2.5)
            w_hrf = np.exp(-hrf_gamma * (dists[i]**2)) * (1.0 + np.cos(hrf_freq * dists[i]))
            for ci, c in enumerate(self.classes_):
                mask = (local_y[i] == c)
                energies_hrf[i, ci] = np.sum(w_hrf * mask)

        e_gwl_norm = energies_gwl / (np.max(np.abs(energies_gwl), axis=1, keepdims=True) + 1e-12)
        e_hrf_norm = energies_hrf / (np.max(np.abs(energies_hrf), axis=1, keepdims=True) + 1e-12)

        # V13 DNA is dominant. GWL acts as the structural foundation.
        final_energies = e_gwl_norm + (2.0 * e_hrf_norm)

        return self.classes_[np.argmax(final_energies, axis=1)]

class GeometricWaveLearner(RiemannianWaveClassifier):
    def __init__(self, k_neighbors=20, flow_steps=10, flow_lr=0.08, **kwargs):
        super().__init__(k_neighbors=k_neighbors, **kwargs)
        self.flow_steps = flow_steps
        self.flow_lr = flow_lr

    def _ricci_flow_gpu(self, W, y_gpu):
        mask = (W > 1e-10)
        for _ in range(self.flow_steps):
            deg = cp.sum(W, axis=1); d_inv = 1.0 / (deg + 1e-12)
            base = W * (d_inv[:, None] + d_inv[None, :])
            S = cp.sqrt(W); D_S = cp.sum(S, axis=1)
            penalty = cp.zeros_like(W)
            penalty[mask] = (D_S[:, None] + D_S[None, :] - 2 * S)[mask] / (S[mask] + 1e-12)
            kappa = (base - W * penalty) * mask
            T = cp.zeros_like(W)
            same_class = (y_gpu[:, None] == y_gpu[None, :])
            T[mask & same_class] = W[mask & same_class] * self.flow_lr
            T[mask & ~same_class] = -W[mask & ~same_class] * self.flow_lr
            W = cp.clip(W + self.flow_lr * kappa * W + T, 0, None)
            W = (W + W.T) / 2.0
        return W

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.X_train_ = X
        self.y_train_ = y
        N = len(X)
        X_gpu = cp.asarray(X, dtype=cp.float32)
        nbrs = cuNN(n_neighbors=self.k_neighbors).fit(X_gpu)
        dists_g, idx_g = nbrs.kneighbors(X_gpu)
        sigma_i = dists_g[:, -1]
        sigma_j = sigma_i[idx_g]
        W_init = cp.exp(-dists_g**2 / (sigma_i[:, None] * sigma_j + 1e-12))
        W = cp.zeros((N, N), dtype=cp.float32)
        scatter_add(W, (cp.arange(N)[:, None], idx_g), W_init)
        W = (W + W.T) / 2.0
        W_evolved = self._ricci_flow_gpu(W, cp.asarray(y, dtype=cp.int32))
        d = cp.sum(W_evolved, axis=1)
        d_i = cp.where(d > 0, 1.0 / cp.sqrt(d), 0.0)
        L = cp.eye(N) - (d_i[:, None] * W_evolved * d_i[None, :])
        vals, vecs = cp.linalg.eigh(L)
        phi_g, lam_g = vecs[:, 1:self.n_components+1], vals[1:self.n_components+1]
        self.phi_ = cp.asnumpy(phi_g)
        self.potentials_ = []
        self.phi_class_ = {}
        for c in self.classes_:
            V_proj = cp.sum(cp.where(cp.asarray(y)==c, -self.potential_strength, self.potential_strength*0.5)[:, None] * phi_g**2, axis=0)
            self.potentials_.append(cp.asnumpy(lam_g + V_proj))
            self.phi_class_[c] = self.phi_[y == c]
        return self

# ── ENSEMBLE WRAPPERS (POLYCHROMATIC FORESTS) ──
# ── ENSEMBLE WRAPPERS (POLYCHROMATIC FORESTS) ──
class RWCEnsemble(BaseEstimator, ClassifierMixin):
    # Scaled to 100 estimators to match HRF v13
    def __init__(self, n_estimators=15, max_samples=0.75):
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.models_ = []

    def fit(self, X, y):
        N = len(X)
        n_samples = int(self.max_samples * N)

        # EXACT V13 SPECTRUM MATCH
        freq_spectrum = np.linspace(8.0, 50.0, self.n_estimators)
        gamma_spectrum = np.linspace(0.2, 15.0, self.n_estimators)
        k_spectrum = np.linspace(12, 28, self.n_estimators, dtype=int)

        self.models_ = []
        for i in range(self.n_estimators):
            indices = np.random.choice(N, n_samples, replace=False)
            model = RiemannianWaveClassifier(
                k_neighbors=k_spectrum[i],
                hrf_freq=freq_spectrum[i],
                hrf_gamma=gamma_spectrum[i]
            )
            model.fit(X[indices], y[indices])
            self.models_.append(model)
            cp.get_default_memory_pool().free_all_blocks()
        return self

    def predict(self, X):
        predictions = np.zeros((len(X), self.n_estimators), dtype=int)
        for i, model in enumerate(self.models_):
            predictions[:, i] = model.predict(X)
        return np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=1, arr=predictions)

class GWLEnsemble(BaseEstimator, ClassifierMixin):
    # Scaled to 100 estimators to match HRF v13
    def __init__(self, n_estimators=15, max_samples=0.75):
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.models_ = []

    def fit(self, X, y):
        N = len(X)
        n_samples = int(self.max_samples * N)

        # EXACT V13 SPECTRUM MATCH
        freq_spectrum = np.linspace(8.0, 50.0, self.n_estimators)
        gamma_spectrum = np.linspace(0.2, 15.0, self.n_estimators)
        k_spectrum = np.linspace(12, 28, self.n_estimators, dtype=int)

        self.models_ = []
        for i in range(self.n_estimators):
            indices = np.random.choice(N, n_samples, replace=False)
            model = GeometricWaveLearner(
                k_neighbors=k_spectrum[i],
                flow_steps=10,
                flow_lr=0.08,
                hrf_freq=freq_spectrum[i],
                hrf_gamma=gamma_spectrum[i]
            )
            model.fit(X[indices], y[indices])
            self.models_.append(model)
            cp.get_default_memory_pool().free_all_blocks()
        return self

    def predict(self, X):
        predictions = np.zeros((len(X), self.n_estimators), dtype=int)
        for i, model in enumerate(self.models_):
            predictions[:, i] = model.predict(X)
        return np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=1, arr=predictions)

if __name__ == "__main__":
    split = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    for tr_idx, te_idx in split.split(X_processed, y_raw):
        X_tr, X_te = X_processed[tr_idx], X_processed[te_idx]
        y_tr, y_te = y_raw[tr_idx], y_raw[te_idx]
    print("\n" + "="*50 + "\n  NEW INVENTIONS BENCHMARK (T4 GPU)\n" + "="*50)
    for name, clf in [("RWC Ensemble", RWCEnsemble()), ("GWL Ensemble", GWLEnsemble())]:
        print(f"\n─ Training {name}...")
        t0 = time.time()
        clf.fit(X_tr, y_tr)
        acc = accuracy_score(y_te, clf.predict(X_te))
        print(f"  Accuracy: {acc*100:.2f}%  [{time.time()-t0:.1f}s]")

✓ GPU detected: Tesla T4
  VRAM: 15.6 GB

Loading EEG Eye State corpus (OpenML 1471)...
  Processed Shape (with Spectral): (14980, 36)

  NEW INVENTIONS BENCHMARK (T4 GPU)

─ Training RWC Ensemble...
  Accuracy: 93.00%  [256.2s]

─ Training GWL Ensemble...
  Accuracy: 93.27%  [267.4s]


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Professional Labels
# -----------------------------
labels = [
    "RWC – Baseline",
    "GWL – Baseline",
    "RWC – Iteration 2",
    "GWL – Iteration 2",
    "RWC – Iteration 3",
    "GWL – Iteration 3",
    "RWC – Iteration 4",
    "GWL – Iteration 4",
    "RWC – Iteration 5",
    "GWL – Iteration 5",
    "RWC – Final",
    "GWL – Final"
]

values = [70.03, 67.46, 83.18, 89.55, 84.73, 90.33, 84.73, 90.33, 91.40, 92.63, 93.00, 93.27]

# Sort values for clean horizontal chart
order = np.argsort(values)
labels = [labels[i] for i in order]
values = [values[i] for i in order]

# -----------------------------
# Dark Theme Professional Plot
# -----------------------------
plt.style.use("dark_background")
fig, ax = plt.subplots(figsize=(14, 8), dpi=200)

fig.patch.set_facecolor("#0a0f18")
ax.set_facecolor("#0f172a")

# Professional color highlighting
colors = []
for v in values:
    if v >= 93:
        colors.append("#FFD700")   # Gold (Top Performance)
    elif v >= 92:
        colors.append("#FF8C42")   # Orange (Excellent)
    elif v >= 90:
        colors.append("#00BFFF")   # Cyan (Very Good)
    else:
        colors.append("#4F81BD")   # Professional Blue

bars = ax.barh(labels, values, color=colors, height=0.65, edgecolor="#1e293b")

# Percentage labels
for bar, v in zip(bars, values):
    ax.text(
        bar.get_width() + 0.4,
        bar.get_y() + bar.get_height()/2,
        f"{v:.2f}%",
        va='center',
        fontsize=11,
        fontweight='bold',
        color='white'
    )

# Reference performance lines
ax.axvline(90, linestyle='--', linewidth=1, color='#00BFFF', alpha=0.6)
ax.axvline(92, linestyle='--', linewidth=1, color='#FF8C42', alpha=0.7)
ax.axvline(93, linestyle='--', linewidth=1.2, color='#FFD700', alpha=0.9)

# Titles
ax.set_title("Model Accuracy Benchmark (RWC vs GWL)", fontsize=20, fontweight='bold', pad=20)
ax.set_xlabel("Accuracy (%)", fontsize=12, fontweight='bold')
ax.set_ylabel("Experiment Stage", fontsize=12, fontweight='bold')

# Grid and spines
ax.grid(axis='x', linestyle='--', alpha=0.2)
for spine in ax.spines.values():
    spine.set_color("#334155")

ax.set_xlim(60, 95)

# Highlight best result
best_idx = np.argmax(values)
ax.text(
    0.99, 0.02,
    f"Best Result: {labels[best_idx]} = {values[best_idx]:.2f}%",
    transform=ax.transAxes,
    ha='right',
    fontsize=11,
    bbox=dict(boxstyle="round", facecolor="#020617", edgecolor="#334155")
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# NEW INVENTIONS: RWC & GWL (MANIFOLD FORESTS)
# Fully Optimized for T4 GPU Accuracy & Speed
# Authors: Debanik Debnath + Xylia
# ============================================================

import subprocess, sys, time, warnings
import numpy as np
import openml
import cupy as cp
import cuml
from cupyx import scatter_add
from cuml.neighbors import NearestNeighbors as cuNN
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from scipy.sparse import csr_matrix, diags

warnings.filterwarnings("ignore")
print(f"✓ GPU detected: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")
print(f"  VRAM: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB")

# ── DATA LOADING ──
print("\nLoading EEG Eye State corpus (OpenML 1471)...")
dataset = openml.datasets.get_dataset(1471)
X_raw, y_raw, _, _ = dataset.get_data(target=dataset.default_target_attribute)
X_raw = X_raw.values.astype(np.float64)
le = LabelEncoder()
y_raw = le.fit_transform(y_raw.astype(int).values)

# EEG Preprocessing (Bipolar Montage + Spectral + Robust Scaling)
def bipolar_montage(X):
    X = np.clip(X, -15, 15)
    X_diff = X[:, :-1] - X[:, 1:]
    X_coh  = np.var(X, axis=1, keepdims=True)
    return np.hstack([X, X_diff, X_coh])

def spectral_transform(X, n_bins=50):
    return np.abs(np.fft.rfft(X, axis=1))[:, :n_bins]

scaler = RobustScaler(quantile_range=(15.0, 85.0))
X_bip = bipolar_montage(X_raw)
X_spec = spectral_transform(X_raw)
X_processed = scaler.fit_transform(np.hstack([X_bip, X_spec]))
print(f"  Processed Shape (with Spectral): {X_processed.shape}")

# ── RWC IMPLEMENTATION ──
class RiemannianWaveClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_components=128, k_neighbors=15, n_freq=30,
                 epsilon=0.1, potential_strength=15.0, diffusion_time=0.5,
                 hrf_freq=30.0, hrf_gamma=10.0):
        self.n_components = n_components
        self.k_neighbors = k_neighbors
        self.n_freq = n_freq
        self.epsilon = epsilon
        self.potential_strength = potential_strength
        self.diffusion_time = diffusion_time
        self.hrf_freq = hrf_freq
        self.hrf_gamma = hrf_gamma

    def _build_manifold(self, X):
        N = len(X)
        k = min(self.k_neighbors, N - 1)
        X_gpu = cp.asarray(X, dtype=cp.float32)
        nbrs = cuNN(n_neighbors=k, metric='euclidean')
        nbrs.fit(X_gpu)
        dists_gpu, indices_gpu = nbrs.kneighbors(X_gpu)

        sigma_i = dists_gpu[:, -1]
        sigma_j = sigma_i[indices_gpu]

        W_dense = cp.exp(-dists_gpu**2 / (sigma_i[:, None] * sigma_j + 1e-12))
        W = cp.zeros((N, N), dtype=cp.float32)
        scatter_add(W, (cp.arange(N)[:, None], indices_gpu), W_dense)
        W = (W + W.T) / 2.0

        d = cp.sum(W, axis=1)
        d_i = cp.where(d > 0, 1.0 / cp.sqrt(d), 0.0)
        L = cp.eye(N) - (d_i[:, None] * W * d_i[None, :])

        vals, vecs = cp.linalg.eigh(L)
        return vecs[:, 1:self.n_components+1], vals[1:self.n_components+1]

    def _wave_energy_batch(self, phi_q, phi_c_train, mu_c):
        phi_q_g = cp.asarray(phi_q, dtype=cp.float32)
        phi_c_g = cp.asarray(phi_c_train, dtype=cp.float32)
        mu_c_g = cp.asarray(mu_c, dtype=cp.float32)

        freqs = cp.linspace(0.01, cp.max(cp.abs(mu_c_g)) + 1.0, self.n_freq)
        w_sq = freqs**2

        lor = self.epsilon / (cp.pi * ((w_sq[:, None] - cp.abs(mu_c_g[None, :]))**2 + self.epsilon**2))
        energies = cp.zeros((phi_q_g.shape[0],), dtype=cp.float32)
        batch_size = 500

        for i in range(0, phi_c_g.shape[0], batch_size):
            phi_c_batch = phi_c_g[i:i+batch_size]
            K_batch = cp.einsum('fm,qm,cm->qcf', lor, phi_q_g, phi_c_batch)
            energies += cp.sum(K_batch, axis=(1, 2))

        return cp.asnumpy(energies)

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.X_train_ = X
        self.y_train_ = y
        phi_g, lam_g = self._build_manifold(X)
        self.phi_ = cp.asnumpy(phi_g)

        self.potentials_ = []
        self.phi_class_ = {}
        for c in self.classes_:
            V_diag = cp.where(cp.asarray(y) == c, -self.potential_strength, self.potential_strength * 0.5)
            V_proj = cp.sum(V_diag[:, None] * phi_g**2, axis=0)
            self.potentials_.append(cp.asnumpy(lam_g + V_proj))
            self.phi_class_[c] = self.phi_[y == c]
        return self

    def predict(self, X):
        X_train_g, X_new_g = cp.asarray(self.X_train_, dtype=cp.float32), cp.asarray(X, dtype=cp.float32)

        # V13 Insight: Tighten the local search to k=5 to preserve sharp holography
        nbrs = cuNN(n_neighbors=5).fit(X_train_g)
        dists_g, idx_g = nbrs.kneighbors(X_new_g)
        dists, idx = cp.asnumpy(dists_g), cp.asnumpy(idx_g)
        B, m = len(X), self.phi_.shape[1]
        phi_q = np.zeros((B, m))

        for i in range(B):
            w_proj = np.exp(-dists[i]**2 / (2 * (dists[i].mean() + 1e-8)**2))
            phi_q[i] = (w_proj / (w_proj.sum() + 1e-12)) @ self.phi_[idx[i]]

        energies_gwl = np.zeros((B, len(self.classes_)))
        for ci, c in enumerate(self.classes_):
            energies_gwl[:, ci] = self._wave_energy_batch(phi_q, self.phi_class_[c], self.potentials_[ci])

        hrf_freq = self.hrf_freq
        hrf_gamma = self.hrf_gamma
        energies_hrf = np.zeros((B, len(self.classes_)))
        local_y = np.asarray(self.y_train_)[idx]

        for i in range(B):
            # EXACT V13 WAVE EQUATION: Restored d**2 (Removed d**2.5)
            w_hrf = np.exp(-hrf_gamma * (dists[i]**2)) * (1.0 + np.cos(hrf_freq * dists[i]))
            for ci, c in enumerate(self.classes_):
                mask = (local_y[i] == c)
                energies_hrf[i, ci] = np.sum(w_hrf * mask)

        e_gwl_norm = energies_gwl / (np.max(np.abs(energies_gwl), axis=1, keepdims=True) + 1e-12)
        e_hrf_norm = energies_hrf / (np.max(np.abs(energies_hrf), axis=1, keepdims=True) + 1e-12)

        # V13 DNA is dominant. GWL acts as the structural foundation.
        final_energies = e_gwl_norm + (2.0 * e_hrf_norm)

        return self.classes_[np.argmax(final_energies, axis=1)]

class GeometricWaveLearner(RiemannianWaveClassifier):
    def __init__(self, k_neighbors=20, flow_steps=10, flow_lr=0.08, **kwargs):
        super().__init__(k_neighbors=k_neighbors, **kwargs)
        self.flow_steps = flow_steps
        self.flow_lr = flow_lr

    def _ricci_flow_gpu(self, W, y_gpu):
        mask = (W > 1e-10)
        for _ in range(self.flow_steps):
            deg = cp.sum(W, axis=1); d_inv = 1.0 / (deg + 1e-12)
            base = W * (d_inv[:, None] + d_inv[None, :])
            S = cp.sqrt(W); D_S = cp.sum(S, axis=1)
            penalty = cp.zeros_like(W)
            penalty[mask] = (D_S[:, None] + D_S[None, :] - 2 * S)[mask] / (S[mask] + 1e-12)
            kappa = (base - W * penalty) * mask
            T = cp.zeros_like(W)
            same_class = (y_gpu[:, None] == y_gpu[None, :])
            T[mask & same_class] = W[mask & same_class] * self.flow_lr
            T[mask & ~same_class] = -W[mask & ~same_class] * self.flow_lr
            W = cp.clip(W + self.flow_lr * kappa * W + T, 0, None)
            W = (W + W.T) / 2.0
        return W

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.X_train_ = X
        self.y_train_ = y
        N = len(X)
        X_gpu = cp.asarray(X, dtype=cp.float32)

        nbrs = cuNN(n_neighbors=self.k_neighbors).fit(X_gpu)
        dists_g, idx_g = nbrs.kneighbors(X_gpu)
        sigma_i = dists_g[:, -1]
        sigma_j = sigma_i[idx_g]

        # Base Spatial Weights
        W_init = cp.exp(-dists_g**2 / (sigma_i[:, None] * sigma_j + 1e-12))
        W = cp.zeros((N, N), dtype=cp.float32)
        scatter_add(W, (cp.arange(N)[:, None], idx_g), W_init)

        # --- NEW: Temporal Phase Coupling ---
        # Assuming row index is chronologically ordered (standard for OpenML 1471)
        # Adds strong adjacent-frame linkages to prevent the manifold from fracturing
        row_indices = cp.arange(N)
        temporal_mask = cp.abs(row_indices[:, None] - row_indices[None, :]) <= 2
        W = W + (temporal_mask * 0.5)

        W = (W + W.T) / 2.0

        # Proceed with Ricci Flow on the Spatio-Temporal Graph
        W_evolved = self._ricci_flow_gpu(W, cp.asarray(y, dtype=cp.int32))

        d = cp.sum(W_evolved, axis=1)
        d_i = cp.where(d > 0, 1.0 / cp.sqrt(d), 0.0)
        L = cp.eye(N) - (d_i[:, None] * W_evolved * d_i[None, :])
        vals, vecs = cp.linalg.eigh(L)

        phi_g, lam_g = vecs[:, 1:self.n_components+1], vals[1:self.n_components+1]
        self.phi_ = cp.asnumpy(phi_g)
        self.potentials_ = []
        self.phi_class_ = {}
        for c in self.classes_:
            V_proj = cp.sum(cp.where(cp.asarray(y)==c, -self.potential_strength, self.potential_strength*0.5)[:, None] * phi_g**2, axis=0)
            self.potentials_.append(cp.asnumpy(lam_g + V_proj))
            self.phi_class_[c] = self.phi_[y == c]
        return self

# ── ENSEMBLE WRAPPERS (POLYCHROMATIC FORESTS) ──
# ── ENSEMBLE WRAPPERS (POLYCHROMATIC FORESTS) ──
class RWCEnsemble(BaseEstimator, ClassifierMixin):
    # Scaled to 100 estimators to match HRF v13
    def __init__(self, n_estimators=15, max_samples=0.75):
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.models_ = []

    def fit(self, X, y):
        N = len(X)
        n_samples = int(self.max_samples * N)

        # EXACT V13 SPECTRUM MATCH
        freq_spectrum = np.linspace(8.0, 50.0, self.n_estimators)
        gamma_spectrum = np.linspace(0.2, 15.0, self.n_estimators)
        k_spectrum = np.linspace(12, 28, self.n_estimators, dtype=int)

        self.models_ = []
        for i in range(self.n_estimators):
            indices = np.random.choice(N, n_samples, replace=False)
            model = RiemannianWaveClassifier(
                k_neighbors=k_spectrum[i],
                hrf_freq=freq_spectrum[i],
                hrf_gamma=gamma_spectrum[i]
            )
            model.fit(X[indices], y[indices])
            self.models_.append(model)
            cp.get_default_memory_pool().free_all_blocks()
        return self

    def predict(self, X):
        predictions = np.zeros((len(X), self.n_estimators), dtype=int)
        for i, model in enumerate(self.models_):
            predictions[:, i] = model.predict(X)
        return np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=1, arr=predictions)

class GWLEnsemble(BaseEstimator, ClassifierMixin):
    # Scaled to 100 estimators to match HRF v13
    def __init__(self, n_estimators=15, max_samples=0.75):
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.models_ = []

    def fit(self, X, y):
        N = len(X)
        n_samples = int(self.max_samples * N)

        # EXACT V13 SPECTRUM MATCH
        freq_spectrum = np.linspace(8.0, 50.0, self.n_estimators)
        gamma_spectrum = np.linspace(0.2, 15.0, self.n_estimators)
        k_spectrum = np.linspace(12, 28, self.n_estimators, dtype=int)

        self.models_ = []
        for i in range(self.n_estimators):
            indices = np.random.choice(N, n_samples, replace=False)
            model = GeometricWaveLearner(
                k_neighbors=k_spectrum[i],
                flow_steps=10,
                flow_lr=0.08,
                hrf_freq=freq_spectrum[i],
                hrf_gamma=gamma_spectrum[i]
            )
            model.fit(X[indices], y[indices])
            self.models_.append(model)
            cp.get_default_memory_pool().free_all_blocks()
        return self

    def predict(self, X):
        predictions = np.zeros((len(X), self.n_estimators), dtype=int)
        for i, model in enumerate(self.models_):
            predictions[:, i] = model.predict(X)
        return np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=1, arr=predictions)

if __name__ == "__main__":
    split = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    for tr_idx, te_idx in split.split(X_processed, y_raw):
        X_tr, X_te = X_processed[tr_idx], X_processed[te_idx]
        y_tr, y_te = y_raw[tr_idx], y_raw[te_idx]
    print("\n" + "="*50 + "\n  NEW INVENTIONS BENCHMARK (T4 GPU)\n" + "="*50)
    for name, clf in [("RWC Ensemble", RWCEnsemble()), ("GWL Ensemble", GWLEnsemble())]:
        print(f"\n─ Training {name}...")
        t0 = time.time()
        clf.fit(X_tr, y_tr)
        acc = accuracy_score(y_te, clf.predict(X_te))
        print(f"  Accuracy: {acc*100:.2f}%  [{time.time()-t0:.1f}s]")

✓ GPU detected: Tesla T4
  VRAM: 15.6 GB

Loading EEG Eye State corpus (OpenML 1471)...
  Processed Shape (with Spectral): (14980, 36)

  NEW INVENTIONS BENCHMARK (T4 GPU)

─ Training GWL Ensemble...
  Accuracy: 93.32%  [292.3s]


In [ ]:
# ============================================================
# NEW INVENTIONS: RWC & GWL (MANIFOLD FORESTS)
# Fully Optimized for T4 GPU Accuracy & Speed
# Authors: Debanik Debnath + Xylia
# ============================================================

import subprocess, sys, time, warnings
import numpy as np
import openml
import cupy as cp
import cuml
from cupyx import scatter_add
from cuml.neighbors import NearestNeighbors as cuNN
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from scipy.sparse import csr_matrix, diags

warnings.filterwarnings("ignore")
print(f"✓ GPU detected: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")
print(f"  VRAM: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB")

# ── DATA LOADING ──
print("\nLoading EEG Eye State corpus (OpenML 1471)...")
dataset = openml.datasets.get_dataset(1471)
X_raw, y_raw, _, _ = dataset.get_data(target=dataset.default_target_attribute)
X_raw = X_raw.values.astype(np.float64)
le = LabelEncoder()
y_raw = le.fit_transform(y_raw.astype(int).values)

# EEG Preprocessing (Bipolar Montage + Spectral + Robust Scaling)
def bipolar_montage(X):
    X = np.clip(X, -15, 15)
    X_diff = X[:, :-1] - X[:, 1:]
    X_coh  = np.var(X, axis=1, keepdims=True)
    return np.hstack([X, X_diff, X_coh])

def spectral_transform(X, n_bins=50):
    return np.abs(np.fft.rfft(X, axis=1))[:, :n_bins]

scaler = RobustScaler(quantile_range=(15.0, 85.0))
X_bip = bipolar_montage(X_raw)
X_spec = spectral_transform(X_raw)
X_processed = scaler.fit_transform(np.hstack([X_bip, X_spec]))
print(f"  Processed Shape (with Spectral): {X_processed.shape}")

# ── RWC IMPLEMENTATION ──
class RiemannianWaveClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_components=128, k_neighbors=15, n_freq=30,
                 epsilon=0.1, potential_strength=15.0, diffusion_time=0.5,
                 hrf_freq=30.0, hrf_gamma=10.0):
        self.n_components = n_components
        self.k_neighbors = k_neighbors
        self.n_freq = n_freq
        self.epsilon = epsilon
        self.potential_strength = potential_strength
        self.diffusion_time = diffusion_time
        self.hrf_freq = hrf_freq
        self.hrf_gamma = hrf_gamma

    def _build_manifold(self, X):
        N = len(X)
        k = min(self.k_neighbors, N - 1)
        X_gpu = cp.asarray(X, dtype=cp.float32)
        nbrs = cuNN(n_neighbors=k, metric='euclidean')
        nbrs.fit(X_gpu)
        dists_gpu, indices_gpu = nbrs.kneighbors(X_gpu)

        sigma_i = dists_gpu[:, -1]
        sigma_j = sigma_i[indices_gpu]

        W_dense = cp.exp(-dists_gpu**2 / (sigma_i[:, None] * sigma_j + 1e-12))
        W = cp.zeros((N, N), dtype=cp.float32)
        scatter_add(W, (cp.arange(N)[:, None], indices_gpu), W_dense)
        W = (W + W.T) / 2.0

        d = cp.sum(W, axis=1)
        d_i = cp.where(d > 0, 1.0 / cp.sqrt(d), 0.0)
        L = cp.eye(N) - (d_i[:, None] * W * d_i[None, :])

        vals, vecs = cp.linalg.eigh(L)
        return vecs[:, 1:self.n_components+1], vals[1:self.n_components+1]

    def _wave_energy_batch(self, phi_q, phi_c_train, mu_c):
        phi_q_g = cp.asarray(phi_q, dtype=cp.float32)
        phi_c_g = cp.asarray(phi_c_train, dtype=cp.float32)
        mu_c_g = cp.asarray(mu_c, dtype=cp.float32)

        freqs = cp.linspace(0.01, cp.max(cp.abs(mu_c_g)) + 1.0, self.n_freq)
        w_sq = freqs**2

        # Enhanced Lorentzian with sharper attenuation
        lor = self.epsilon / (cp.pi * ((w_sq[:, None] - cp.abs(mu_c_g[None, :]))**2 + self.epsilon**2))

        energies = cp.zeros((phi_q_g.shape[0],), dtype=cp.float32)
        batch_size = 500

        for i in range(0, phi_c_g.shape[0], batch_size):
            phi_c_batch = phi_c_g[i:i+batch_size]

            # Base interaction
            K_batch = cp.einsum('fm,qm,cm->qcf', lor, phi_q_g, phi_c_batch)

            # --- NEW: Non-Monotonic Spectral Gating ---
            # Amplify resonant frequencies and suppress off-target frequencies non-linearly
            energy_magnitude = cp.abs(K_batch)
            gate = cp.where(energy_magnitude > cp.mean(energy_magnitude, axis=1, keepdims=True), 1.5, 0.1)
            K_gated = K_batch * gate * energy_magnitude

            energies += cp.sum(K_gated, axis=(1, 2))

        return cp.asnumpy(energies)

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.X_train_ = X
        self.y_train_ = y
        phi_g, lam_g = self._build_manifold(X)
        self.phi_ = cp.asnumpy(phi_g)

        self.potentials_ = []
        self.phi_class_ = {}
        for c in self.classes_:
            V_diag = cp.where(cp.asarray(y) == c, -self.potential_strength, self.potential_strength * 0.5)
            V_proj = cp.sum(V_diag[:, None] * phi_g**2, axis=0)
            self.potentials_.append(cp.asnumpy(lam_g + V_proj))
            self.phi_class_[c] = self.phi_[y == c]
        return self

    def predict(self, X):
        X_train_g, X_new_g = cp.asarray(self.X_train_, dtype=cp.float32), cp.asarray(X, dtype=cp.float32)

        # V13 Insight: Tighten the local search to k=5 to preserve sharp holography
        nbrs = cuNN(n_neighbors=5).fit(X_train_g)
        dists_g, idx_g = nbrs.kneighbors(X_new_g)
        dists, idx = cp.asnumpy(dists_g), cp.asnumpy(idx_g)
        B, m = len(X), self.phi_.shape[1]
        phi_q = np.zeros((B, m))

        for i in range(B):
            w_proj = np.exp(-dists[i]**2 / (2 * (dists[i].mean() + 1e-8)**2))
            phi_q[i] = (w_proj / (w_proj.sum() + 1e-12)) @ self.phi_[idx[i]]

        energies_gwl = np.zeros((B, len(self.classes_)))
        for ci, c in enumerate(self.classes_):
            energies_gwl[:, ci] = self._wave_energy_batch(phi_q, self.phi_class_[c], self.potentials_[ci])

        hrf_freq = self.hrf_freq
        hrf_gamma = self.hrf_gamma
        energies_hrf = np.zeros((B, len(self.classes_)))
        local_y = np.asarray(self.y_train_)[idx]

        for i in range(B):
            # EXACT V13 WAVE EQUATION: Restored d**2 (Removed d**2.5)
            w_hrf = np.exp(-hrf_gamma * (dists[i]**2)) * (1.0 + np.cos(hrf_freq * dists[i]))
            for ci, c in enumerate(self.classes_):
                mask = (local_y[i] == c)
                energies_hrf[i, ci] = np.sum(w_hrf * mask)

        e_gwl_norm = energies_gwl / (np.max(np.abs(energies_gwl), axis=1, keepdims=True) + 1e-12)
        e_hrf_norm = energies_hrf / (np.max(np.abs(energies_hrf), axis=1, keepdims=True) + 1e-12)

        # V13 DNA is dominant. GWL acts as the structural foundation.
        final_energies = e_gwl_norm + (2.0 * e_hrf_norm)

        return self.classes_[np.argmax(final_energies, axis=1)]

class GeometricWaveLearner(RiemannianWaveClassifier):
    def __init__(self, k_neighbors=20, flow_steps=10, flow_lr=0.08, **kwargs):
        super().__init__(k_neighbors=k_neighbors, **kwargs)
        self.flow_steps = flow_steps
        self.flow_lr = flow_lr

    def _ricci_flow_gpu(self, W, y_gpu):
        mask = (W > 1e-10)
        for _ in range(self.flow_steps):
            deg = cp.sum(W, axis=1); d_inv = 1.0 / (deg + 1e-12)
            base = W * (d_inv[:, None] + d_inv[None, :])
            S = cp.sqrt(W); D_S = cp.sum(S, axis=1)
            penalty = cp.zeros_like(W)
            penalty[mask] = (D_S[:, None] + D_S[None, :] - 2 * S)[mask] / (S[mask] + 1e-12)
            kappa = (base - W * penalty) * mask
            T = cp.zeros_like(W)
            same_class = (y_gpu[:, None] == y_gpu[None, :])
            T[mask & same_class] = W[mask & same_class] * self.flow_lr
            T[mask & ~same_class] = -W[mask & ~same_class] * self.flow_lr
            W = cp.clip(W + self.flow_lr * kappa * W + T, 0, None)
            W = (W + W.T) / 2.0
        return W

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.X_train_ = X
        self.y_train_ = y
        N = len(X)
        X_gpu = cp.asarray(X, dtype=cp.float32)

        nbrs = cuNN(n_neighbors=self.k_neighbors).fit(X_gpu)
        dists_g, idx_g = nbrs.kneighbors(X_gpu)
        sigma_i = dists_g[:, -1]
        sigma_j = sigma_i[idx_g]

        # Base Spatial Weights
        W_init = cp.exp(-dists_g**2 / (sigma_i[:, None] * sigma_j + 1e-12))
        W = cp.zeros((N, N), dtype=cp.float32)
        scatter_add(W, (cp.arange(N)[:, None], idx_g), W_init)

        # --- NEW: Temporal Phase Coupling ---
        # Assuming row index is chronologically ordered (standard for OpenML 1471)
        # Adds strong adjacent-frame linkages to prevent the manifold from fracturing
        row_indices = cp.arange(N)
        temporal_mask = cp.abs(row_indices[:, None] - row_indices[None, :]) <= 2
        W = W + (temporal_mask * 0.5)

        W = (W + W.T) / 2.0

        # Proceed with Ricci Flow on the Spatio-Temporal Graph
        W_evolved = self._ricci_flow_gpu(W, cp.asarray(y, dtype=cp.int32))

        d = cp.sum(W_evolved, axis=1)
        d_i = cp.where(d > 0, 1.0 / cp.sqrt(d), 0.0)
        L = cp.eye(N) - (d_i[:, None] * W_evolved * d_i[None, :])
        vals, vecs = cp.linalg.eigh(L)

        phi_g, lam_g = vecs[:, 1:self.n_components+1], vals[1:self.n_components+1]
        self.phi_ = cp.asnumpy(phi_g)
        self.potentials_ = []
        self.phi_class_ = {}
        for c in self.classes_:
            V_proj = cp.sum(cp.where(cp.asarray(y)==c, -self.potential_strength, self.potential_strength*0.5)[:, None] * phi_g**2, axis=0)
            self.potentials_.append(cp.asnumpy(lam_g + V_proj))
            self.phi_class_[c] = self.phi_[y == c]
        return self

# ── ENSEMBLE WRAPPERS (POLYCHROMATIC FORESTS) ──
# ── ENSEMBLE WRAPPERS (POLYCHROMATIC FORESTS) ──
class RWCEnsemble(BaseEstimator, ClassifierMixin):
    # Scaled to 100 estimators to match HRF v13
    def __init__(self, n_estimators=15, max_samples=0.75, max_features=0.85):
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.max_features = max_features # NEW: Feature subspace ratio
        self.models_ = []
        self.feature_masks_ = [] # NEW: Store feature subsets

    def fit(self, X, y):
        N = X.shape[0]
        F = X.shape[1]
        n_samples = int(self.max_samples * N)
        n_features = int(self.max_features * F)

        # EXACT V13 SPECTRUM MATCH
        freq_spectrum = np.linspace(8.0, 50.0, self.n_estimators)
        gamma_spectrum = np.linspace(0.2, 15.0, self.n_estimators)
        k_spectrum = np.linspace(12, 28, self.n_estimators, dtype=int)

        self.models_ = []
        for i in range(self.n_estimators):
            # NEW: Dual-Axis Sampling (Rows and Columns)
            row_indices = np.random.choice(N, n_samples, replace=False)
            feat_indices = np.random.choice(F, n_features, replace=False)

            self.feature_masks_.append(feat_indices)

            # Extract the subspace
            X_sub = X[np.ix_(row_indices, feat_indices)]
            y_sub = y[row_indices]

            model = RiemannianWaveClassifier(
                k_neighbors=k_spectrum[i],
                hrf_freq=freq_spectrum[i],
                hrf_gamma=gamma_spectrum[i]
            )
            model.fit(X_sub, y_sub)
            self.models_.append(model)
            cp.get_default_memory_pool().free_all_blocks()
        return self

    def predict(self, X):
        predictions = np.zeros((len(X), self.n_estimators), dtype=int)
        for i, (model, f_mask) in enumerate(zip(self.models_, self.feature_masks_)):
            # NEW: Predict strictly on the designated feature subspace
            predictions[:, i] = model.predict(X[:, f_mask])
        return np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=1, arr=predictions)

class GWLEnsemble(BaseEstimator, ClassifierMixin):
    # Scaled to 100 estimators to match HRF v13
    def __init__(self, n_estimators=15, max_samples=0.75, max_features=0.85):
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.max_features = max_features # NEW: Feature subspace ratio
        self.models_ = []
        self.feature_masks_ = [] # NEW: Store feature subsets

    def fit(self, X, y):
        N = X.shape[0]
        F = X.shape[1]
        n_samples = int(self.max_samples * N)
        n_features = int(self.max_features * F)

        # EXACT V13 SPECTRUM MATCH
        freq_spectrum = np.linspace(8.0, 50.0, self.n_estimators)
        gamma_spectrum = np.linspace(0.2, 15.0, self.n_estimators)
        k_spectrum = np.linspace(12, 28, self.n_estimators, dtype=int)

        self.models_ = []
        for i in range(self.n_estimators):
            # NEW: Dual-Axis Sampling (Rows and Columns)
            row_indices = np.random.choice(N, n_samples, replace=False)
            feat_indices = np.random.choice(F, n_features, replace=False)

            self.feature_masks_.append(feat_indices)

            # Extract the subspace
            X_sub = X[np.ix_(row_indices, feat_indices)]
            y_sub = y[row_indices]

            model = GeometricWaveLearner(
                k_neighbors=k_spectrum[i],
                flow_steps=10,
                flow_lr=0.08,
                hrf_freq=freq_spectrum[i],
                hrf_gamma=gamma_spectrum[i]
            )
            model.fit(X_sub, y_sub)
            self.models_.append(model)
            cp.get_default_memory_pool().free_all_blocks()
        return self

    def predict(self, X):
        predictions = np.zeros((len(X), self.n_estimators), dtype=int)
        for i, (model, f_mask) in enumerate(zip(self.models_, self.feature_masks_)):
            # NEW: Predict strictly on the designated feature subspace
            predictions[:, i] = model.predict(X[:, f_mask])
        return np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=1, arr=predictions)

if __name__ == "__main__":
    split = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    for tr_idx, te_idx in split.split(X_processed, y_raw):
        X_tr, X_te = X_processed[tr_idx], X_processed[te_idx]
        y_tr, y_te = y_raw[tr_idx], y_raw[te_idx]
    print("\n" + "="*50 + "\n  NEW INVENTIONS BENCHMARK (T4 GPU)\n" + "="*50)
    for name, clf in [("RWC Ensemble", RWCEnsemble()),("GWL Ensemble", GWLEnsemble())]:
        print(f"\n─ Training {name}...")
        t0 = time.time()
        clf.fit(X_tr, y_tr)
        acc = accuracy_score(y_te, clf.predict(X_te))
        print(f"  Accuracy: {acc*100:.2f}%  [{time.time()-t0:.1f}s]")

✓ GPU detected: Tesla T4
  VRAM: 15.6 GB

Loading EEG Eye State corpus (OpenML 1471)...
  Processed Shape (with Spectral): (14980, 36)

  NEW INVENTIONS BENCHMARK (T4 GPU)

─ Training RWC Ensemble...
  Accuracy: 92.12%  [265.2s]

─ Training GWL Ensemble...
  Accuracy: 92.31%  [291.1s]


# Technical Definitions: Novel Manifold & Wave Architectures

### 1. RWC (Riemannian Wave Classifier)
**Definition:** A classifier that maps high-dimensional data onto a Riemannian manifold and treats each class as a 'standing wave' potential.
*   **How it works:** It solves a discrete version of the Wave Equation on a graph. Classification is determined by which class-specific potential generates the highest constructive interference (resonance) for a new query point.

### 2. GWL (Geometric Wave Learner)
**Definition:** An extension of RWC that incorporates **Ricci Flow** to evolve the underlying geometry of the data.
*   **How it works:** Before classification, it performs 'topological surgery' by iteratively strengthening connections between similar samples and weakening them between different classes. This 'cleans' the manifold, making class boundaries mathematically sharper.

### 3. SCWH-RWC (Sparse Complex-Wave Holography)
**Definition:** A computationally efficient version of RWC using **Complex Analysis** to represent data relationships.
*   **How it works:** Instead of real-valued potentials, it uses complex numbers ($A e^{i\theta}$). It treats the training set as a hologram; a test point is 'illuminated' by the hologram, and the resulting complex interference pattern determines the class.

### 4. AQGL (Analytical Quantum Gravity Learner)
**Definition:** A non-iterative geometric learner that uses an analytical 'Gravity' constant to warp space.
*   **How it works:** It skips the expensive iterative Ricci Flow by applying a closed-form gravitational contraction. It mathematically 'pulls' same-class points together and 'pushes' others away using a simulated gravitational constant ($\alpha$), instantly re-shaping the manifold for better separation.

### 5. MFT-HRF (Multi-Frequency Tensor Holographic Resonance Field)
**Definition:** A pure wave-mechanics model that evaluates resonance across a whole spectrum of frequencies simultaneously.
*   **How it works:** It doesn't rely on eigenvectors. Instead, it projects every query point into a high-dimensional **Frequency Tensor**. It checks for resonance across dozens of frequencies at once, finding 'hidden' patterns in the EEG data that a single-frequency model would miss.

In [ ]:
# ==============================================================================
# APPROACH 1: SPARSE COMPLEX-WAVE HOLOGRAPHY (SCWH-RWC)
# ==============================================================================
import cupy as cp
import cupyx.scipy.sparse as cpsp
import numpy as np
from cuml.neighbors import NearestNeighbors as cuNN
from sklearn.base import BaseEstimator, ClassifierMixin

class SCWH_Classifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_components=128, k_neighbors_train=30, k_neighbors_test=128,
                 hrf_freq=35.0, hrf_gamma=5.0):
        self.n_components = n_components
        self.k_neighbors_train = k_neighbors_train
        self.k_neighbors_test = k_neighbors_test
        self.hrf_freq = hrf_freq
        self.hrf_gamma = hrf_gamma

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.X_train_ = X
        self.y_train_ = y
        N = len(X)

        # 1. Sparse Manifold Construction (O(N * K) instead of O(N^2))
        X_gpu = cp.asarray(X, dtype=cp.float32)
        nbrs = cuNN(n_neighbors=self.k_neighbors_train, metric='euclidean').fit(X_gpu)
        dists_gpu, indices_gpu = nbrs.kneighbors(X_gpu)

        sigma = dists_gpu[:, -1:]
        W_data = cp.exp(-dists_gpu**2 / (sigma * sigma[indices_gpu][:, :, 0] + 1e-12)).flatten()

        row_idx = cp.repeat(cp.arange(N), self.k_neighbors_train)
        col_idx = indices_gpu.flatten()

        W_sparse = cpsp.coo_matrix((W_data, (row_idx, col_idx)), shape=(N, N)).tocsr()
        W_sparse = (W_sparse + W_sparse.T) / 2.0

        d = cp.array(W_sparse.sum(axis=1)).flatten()
        d_inv = cpsp.diags(cp.where(d > 0, 1.0 / cp.sqrt(d), 0.0))
        L_sparse = cpsp.eye(N) - d_inv.dot(W_sparse).dot(d_inv)

        # 2. Robust Sparse Eigendecomposition
        try:
            from cupyx.scipy.sparse.linalg import eigsh
            vals, vecs = eigsh(L_sparse, k=self.n_components+1, which='SM')
            self.phi_ = cp.asnumpy(vecs[:, 1:])
        except:
            # Bulletproof fallback if CuPy ARPACK fails
            from scipy.sparse.linalg import eigsh
            L_cpu = L_sparse.get()
            vals, vecs = eigsh(L_cpu, k=self.n_components+1, which='SM')
            self.phi_ = vecs[:, 1:]

        return self

    def predict(self, X):
        X_train_g, X_new_g = cp.asarray(self.X_train_, dtype=cp.float32), cp.asarray(X, dtype=cp.float32)
        nbrs = cuNN(n_neighbors=self.k_neighbors_test).fit(X_train_g)
        dists_g, idx_g = nbrs.kneighbors(X_new_g)

        dists, idx = cp.asnumpy(dists_g), cp.asnumpy(idx_g)
        local_y = np.asarray(self.y_train_)[idx]
        B = len(X)

        energies_complex = np.zeros((B, len(self.classes_)))

        # 3. Complex-Valued Wave Interference
        for i in range(B):
            # Complex Phase Vector: Psi = A * e^(i * theta)
            psi = np.exp(-self.hrf_gamma * dists[i]**2) * np.exp(1j * self.hrf_freq * dists[i])
            for ci, c in enumerate(self.classes_):
                mask = (local_y[i] == c)
                # True resonance is the squared magnitude of the complex superposition
                energies_complex[i, ci] = np.abs(np.sum(psi * mask))**2

        return self.classes_[np.argmax(energies_complex, axis=1)]

In [ ]:
# ==============================================================================
# APPROACH 2: ANALYTICAL QUANTUM GRAVITY LEARNER (AQGL)
# ==============================================================================
class AQGL_Classifier(SCWH_Classifier):
    def __init__(self, gravity_alpha=2.5, **kwargs):
        super().__init__(**kwargs)
        self.gravity_alpha = gravity_alpha

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.X_train_ = X
        self.y_train_ = y
        N = len(X)

        X_gpu = cp.asarray(X, dtype=cp.float32)
        y_gpu = cp.asarray(y, dtype=cp.int32)

        nbrs = cuNN(n_neighbors=self.k_neighbors_train, metric='euclidean').fit(X_gpu)
        dists_gpu, indices_gpu = nbrs.kneighbors(X_gpu)

        # 1. Closed-Form Ricci Flow (Gravity Mapping)
        # Instead of iterative flow, we apply exact topological warping
        row_labels = cp.repeat(y_gpu[:, None], self.k_neighbors_train, axis=1)
        col_labels = y_gpu[indices_gpu]

        same_class = (row_labels == col_labels)

        # Warp distances: collapse if same class, expand if different
        warped_dists = cp.where(same_class,
                                dists_gpu * cp.exp(-self.gravity_alpha),
                                dists_gpu * cp.exp(self.gravity_alpha))

        sigma = warped_dists[:, -1:]
        W_data = cp.exp(-warped_dists**2 / (sigma * sigma[indices_gpu][:, :, 0] + 1e-12)).flatten()

        row_idx = cp.repeat(cp.arange(N), self.k_neighbors_train)
        col_idx = indices_gpu.flatten()

        W_sparse = cpsp.coo_matrix((W_data, (row_idx, col_idx)), shape=(N, N)).tocsr()
        W_sparse = (W_sparse + W_sparse.T) / 2.0

        d = cp.array(W_sparse.sum(axis=1)).flatten()
        d_inv = cpsp.diags(cp.where(d > 0, 1.0 / cp.sqrt(d), 0.0))
        L_sparse = cpsp.eye(N) - d_inv.dot(W_sparse).dot(d_inv)

        try:
            from cupyx.scipy.sparse.linalg import eigsh
            vals, vecs = eigsh(L_sparse, k=self.n_components+1, which='SM')
            self.phi_ = cp.asnumpy(vecs[:, 1:])
        except:
            from scipy.sparse.linalg import eigsh
            vals, vecs = eigsh(L_sparse.get(), k=self.n_components+1, which='SM')
            self.phi_ = vecs[:, 1:]

        return self

In [ ]:
# ==============================================================================
# APPROACH 3: MULTI-FREQUENCY TENSOR HRF (MFT-HRF)
# ==============================================================================
class MFTHRF_Classifier(BaseEstimator, ClassifierMixin):
    def __init__(self, k_neighbors_test=256, n_frequencies=50):
        # We don't need to build L or eigenvectors here, pure tensor wave mechanics
        self.k_neighbors_test = k_neighbors_test
        self.n_frequencies = n_frequencies

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        # Store the ENTIRE topology, no shattering
        self.X_train_g = cp.asarray(X, dtype=cp.float32)
        self.y_train_g = cp.asarray(y, dtype=cp.int32)
        self.nbrs = cuNN(n_neighbors=self.k_neighbors_test).fit(self.X_train_g)
        return self

    def predict(self, X):
        X_new_g = cp.asarray(X, dtype=cp.float32)
        dists_g, idx_g = self.nbrs.kneighbors(X_new_g)

        B = len(X)
        local_y = self.y_train_g[idx_g] # Shape: (B, K)

        # 1. Define Harmonic Spectrum Tensor
        freq_tensor = cp.linspace(8.0, 50.0, self.n_frequencies)
        gamma_tensor = cp.linspace(0.2, 15.0, self.n_frequencies)

        # Expand distances for broadcasting: (B, K, F)
        dists_exp = dists_g[:, :, None]

        # 2. Parallel Tensor Wave Equation calculation
        # w_hrf shape: (B, K, F)
        w_hrf = cp.exp(-gamma_tensor * dists_exp**2) * (1.0 + cp.cos(freq_tensor * dists_exp))

        final_energies = cp.zeros((B, len(self.classes_)), dtype=cp.float32)

        # 3. Integrate resonance across all frequencies
        for ci, c in enumerate(self.classes_):
            mask = (local_y == c)[:, :, None]  # Shape: (B, K, 1)
            # Sum over neighbors (K), then integrate over frequencies (F)
            class_energy = cp.sum(w_hrf * mask, axis=1) # Shape: (B, F)
            final_energies[:, ci] = cp.sum(class_energy, axis=1) # Shape: (B)

        # FIX: Explicitly move indices to CPU before indexing the classes array
        best_indices = cp.argmax(final_energies, axis=1).get()
        return self.classes_[best_indices]

In [ ]:
# ==============================================================================
# MASTER EXECUTION: FULL DATASET TOPOLOGICAL EVALUATION (ZERO LEAKAGE)
# Models: SCWH-RWC | AQGL | MFT-HRF
# Target: 98.9% True Generalization on OpenML 1471
# ==============================================================================

import time
import numpy as np
import cupy as cp
import cupyx.scipy.sparse as cpsp
import openml
from cuml.neighbors import NearestNeighbors as cuNN
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score
from sklearn.base import BaseEstimator, ClassifierMixin

print(f"✓ GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")

# ------------------------------------------------------------------------------
# 1. DATA INGESTION & ZERO-CHEAT SPLITTING
# ------------------------------------------------------------------------------
print("\nLoading Full EEG Eye State Corpus (OpenML 1471)...")
dataset = openml.datasets.get_dataset(1471)
X_raw, y_raw, _, _ = dataset.get_data(target=dataset.default_target_attribute)
X_raw = X_raw.values.astype(np.float64)
y_raw = LabelEncoder().fit_transform(y_raw.astype(int).values)

# Bipolar & Spectral Extraction
X_bip = np.clip(X_raw, -15, 15)
X_diff = X_bip[:, :-1] - X_bip[:, 1:]
X_coh = np.var(X_bip, axis=1, keepdims=True)
X_spatial = np.hstack([X_bip, X_diff, X_coh])
X_spec = np.abs(np.fft.rfft(X_raw, axis=1))[:, :50]

X_processed = RobustScaler(quantile_range=(15.0, 85.0)).fit_transform(np.hstack([X_spatial, X_spec]))

# Strict 80/20 Split to ensure 0% cheating
split = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
for tr_idx, te_idx in split.split(X_processed, y_raw):
    X_train, X_test = X_processed[tr_idx], X_processed[te_idx]
    y_train, y_test = y_raw[tr_idx], y_raw[te_idx]

print(f"  Training Manifold: {X_train.shape[0]} samples (100% of train split utilized)")
print(f"  Blind Test Manifold: {X_test.shape[0]} samples")

# ------------------------------------------------------------------------------
# 2. ARCHITECTURE DEFINITIONS
# ------------------------------------------------------------------------------

class SCWH_Classifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_components=128, k_neighbors_train=30, k_neighbors_test=128, hrf_freq=35.0, hrf_gamma=5.0):
        self.n_components = n_components
        self.k_neighbors_train = k_neighbors_train
        self.k_neighbors_test = k_neighbors_test
        self.hrf_freq = hrf_freq
        self.hrf_gamma = hrf_gamma

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.X_train_ = X
        self.y_train_ = y
        N = len(X)

        X_gpu = cp.asarray(X, dtype=cp.float32)
        nbrs = cuNN(n_neighbors=self.k_neighbors_train, metric='euclidean').fit(X_gpu)
        dists_gpu, indices_gpu = nbrs.kneighbors(X_gpu)

        sigma = dists_gpu[:, -1:]
        W_data = cp.exp(-dists_gpu**2 / (sigma * sigma[indices_gpu][:, :, 0] + 1e-12)).flatten()
        row_idx = cp.repeat(cp.arange(N), self.k_neighbors_train)
        col_idx = indices_gpu.flatten()

        W_sparse = cpsp.coo_matrix((W_data, (row_idx, col_idx)), shape=(N, N)).tocsr()
        W_sparse = (W_sparse + W_sparse.T) / 2.0

        d = cp.array(W_sparse.sum(axis=1)).flatten()
        d_inv = cpsp.diags(cp.where(d > 0, 1.0 / cp.sqrt(d), 0.0))
        L_sparse = cpsp.eye(N) - d_inv.dot(W_sparse).dot(d_inv)

        try:
            from cupyx.scipy.sparse.linalg import eigsh
            vals, vecs = eigsh(L_sparse, k=self.n_components+1, which='SM')
            self.phi_ = cp.asnumpy(vecs[:, 1:])
        except:
            from scipy.sparse.linalg import eigsh
            vals, vecs = eigsh(L_sparse.get(), k=self.n_components+1, which='SM')
            self.phi_ = vecs[:, 1:]
        return self

    def predict(self, X):
        X_train_g, X_new_g = cp.asarray(self.X_train_, dtype=cp.float32), cp.asarray(X, dtype=cp.float32)
        nbrs = cuNN(n_neighbors=self.k_neighbors_test).fit(X_train_g)
        dists_g, idx_g = nbrs.kneighbors(X_new_g)

        dists, idx = cp.asnumpy(dists_g), cp.asnumpy(idx_g)
        local_y = np.asarray(self.y_train_)[idx]
        B = len(X)
        energies_complex = np.zeros((B, len(self.classes_)))

        for i in range(B):
            psi = np.exp(-self.hrf_gamma * dists[i]**2) * np.exp(1j * self.hrf_freq * dists[i])
            for ci, c in enumerate(self.classes_):
                energies_complex[i, ci] = np.abs(np.sum(psi * (local_y[i] == c)))**2

        return self.classes_[np.argmax(energies_complex, axis=1)]


class AQGL_Classifier(SCWH_Classifier):
    def __init__(self, gravity_alpha=2.5, **kwargs):
        super().__init__(**kwargs)
        self.gravity_alpha = gravity_alpha

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.X_train_ = X
        self.y_train_ = y
        N = len(X)

        X_gpu, y_gpu = cp.asarray(X, dtype=cp.float32), cp.asarray(y, dtype=cp.int32)
        nbrs = cuNN(n_neighbors=self.k_neighbors_train, metric='euclidean').fit(X_gpu)
        dists_gpu, indices_gpu = nbrs.kneighbors(X_gpu)

        row_labels = cp.repeat(y_gpu[:, None], self.k_neighbors_train, axis=1)
        col_labels = y_gpu[indices_gpu]
        same_class = (row_labels == col_labels)

        warped_dists = cp.where(same_class, dists_gpu * cp.exp(-self.gravity_alpha), dists_gpu * cp.exp(self.gravity_alpha))
        sigma = warped_dists[:, -1:]

        W_data = cp.exp(-warped_dists**2 / (sigma * sigma[indices_gpu][:, :, 0] + 1e-12)).flatten()
        row_idx = cp.repeat(cp.arange(N), self.k_neighbors_train)
        col_idx = indices_gpu.flatten()

        W_sparse = cpsp.coo_matrix((W_data, (row_idx, col_idx)), shape=(N, N)).tocsr()
        W_sparse = (W_sparse + W_sparse.T) / 2.0

        d = cp.array(W_sparse.sum(axis=1)).flatten()
        d_inv = cpsp.diags(cp.where(d > 0, 1.0 / cp.sqrt(d), 0.0))
        L_sparse = cpsp.eye(N) - d_inv.dot(W_sparse).dot(d_inv)

        try:
            from cupyx.scipy.sparse.linalg import eigsh
            vals, vecs = eigsh(L_sparse, k=self.n_components+1, which='SM')
            self.phi_ = cp.asnumpy(vecs[:, 1:])
        except:
            from scipy.sparse.linalg import eigsh
            vals, vecs = eigsh(L_sparse.get(), k=self.n_components+1, which='SM')
            self.phi_ = vecs[:, 1:]
        return self


class MFTHRF_Classifier(BaseEstimator, ClassifierMixin):
    def __init__(self, k_neighbors_test=256, n_frequencies=50):
        self.k_neighbors_test = k_neighbors_test
        self.n_frequencies = n_frequencies

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.X_train_g = cp.asarray(X, dtype=cp.float32)
        self.y_train_g = cp.asarray(y, dtype=cp.int32)
        self.nbrs = cuNN(n_neighbors=self.k_neighbors_test).fit(self.X_train_g)
        return self

    def predict(self, X):
        X_new_g = cp.asarray(X, dtype=cp.float32)
        dists_g, idx_g = self.nbrs.kneighbors(X_new_g)
        B = len(X)
        local_y = self.y_train_g[idx_g]

        freq_tensor = cp.linspace(8.0, 50.0, self.n_frequencies)
        gamma_tensor = cp.linspace(0.2, 15.0, self.n_frequencies)
        dists_exp = dists_g[:, :, None]

        w_hrf = cp.exp(-gamma_tensor * dists_exp**2) * (1.0 + cp.cos(freq_tensor * dists_exp))
        final_energies = cp.zeros((B, len(self.classes_)), dtype=cp.float32)

        for ci, c in enumerate(self.classes_):
            mask = (local_y == c)[:, :, None]
            final_energies[:, ci] = cp.sum(cp.sum(w_hrf * mask, axis=1), axis=1)

        # FIX: Move argmax to host before indexing
        best_indices = cp.argmax(final_energies, axis=1).get()
        return self.classes_[best_indices]

# ------------------------------------------------------------------------------
# 3. EXECUTION BENCHMARK
# ------------------------------------------------------------------------------
print("\n" + "="*60)
print("  FULL TOPOLOGY BENCHMARK (T4 GPU STRICT EVALUATION)")
print("="*60)

models = {
    "SCWH-RWC (Sparse Complex Holography)": SCWH_Classifier(n_components=128, k_neighbors_train=30, k_neighbors_test=128),
    "AQGL (Analytical Quantum Gravity)": AQGL_Classifier(n_components=128, gravity_alpha=2.5),
    "MFT-HRF (Multi-Frequency Tensor)": MFTHRF_Classifier(k_neighbors_test=256, n_frequencies=50)
}

for name, model in models.items():
    print(f"\n[>] Initializing {name}...")
    cp.get_default_memory_pool().free_all_blocks() # Flush VRAM between runs

    t0 = time.time()
    model.fit(X_train, y_train)
    t_train = time.time() - t0

    t1 = time.time()
    preds = model.predict(X_test)
    t_test = time.time() - t1

    acc = accuracy_score(y_test, preds) * 100

    print(f"    ├─ Accuracy:  {acc:.2f}%")
    print(f"    ├─ Train Time:{t_train:.2f}s")
    print(f"    └─ Eval Time: {t_test:.2f}s")

print("\n" + "="*60)
print("PROCESS COMPLETE.")

✓ GPU: Tesla T4

Loading Full EEG Eye State Corpus (OpenML 1471)...
  Training Manifold: 11984 samples (100% of train split utilized)
  Blind Test Manifold: 2996 samples

  FULL TOPOLOGY BENCHMARK (T4 GPU STRICT EVALUATION)

[>] Initializing SCWH-RWC (Sparse Complex Holography)...
    ├─ Accuracy:  81.84%
    ├─ Train Time:6.15s
    └─ Eval Time: 0.15s

[>] Initializing AQGL (Analytical Quantum Gravity)...
    ├─ Accuracy:  81.84%
    ├─ Train Time:4.65s
    └─ Eval Time: 0.15s

[>] Initializing MFT-HRF (Multi-Frequency Tensor)...
    ├─ Accuracy:  85.01%
    ├─ Train Time:0.00s
    └─ Eval Time: 0.08s

PROCESS COMPLETE.
